# GSM8K Evaluation

Reads `evaluation/config.yaml` and evaluates the specified model on GSM8K.

**To evaluate a different model or checkpoint**, only edit `config.yaml`:
- `model_id` — HuggingFace model ID
- `lora_checkpoint` — path to LoRA adapter, or `null` for base model only
- `output_prefix` — prefix for the output JSON filename

Results are saved to `evaluation/results/{output_prefix}_{split}.json`.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

# Unzip project
!rm -rf /content/project
!unzip -q "/content/drive/MyDrive/project_sft.zip" -d /content/project
!ls /content/project/project_sft

In [ ]:
!pip install -q transformers peft datasets accelerate tqdm pyyaml
!pip uninstall -y torchao  # avoid version conflict

## 1. Load Config

In [ ]:
import sys
import yaml
from pathlib import Path

PROJECT_ROOT = Path('/content/project/project_sft')
EVAL_DIR     = PROJECT_ROOT / 'evaluation'

# Add evaluation/ to path so we can import evaluate.py and answer_extractor.py
sys.path.insert(0, str(EVAL_DIR))

config_path = EVAL_DIR / 'config.yaml'
cfg = yaml.safe_load(config_path.read_text())

MODEL_ID         = cfg['model_id']
LORA_CHECKPOINT  = cfg.get('lora_checkpoint')   # None = base model only
DATASET_NAME     = cfg['dataset_name']
DATASET_CONFIG   = cfg['dataset_config']
SPLITS           = cfg['splits']
MAX_NEW_TOKENS   = cfg['max_new_tokens']
EVAL_BATCH_SIZE  = cfg['eval_batch_size']
OUTPUT_DIR       = PROJECT_ROOT / cfg['output_dir']
OUTPUT_PREFIX    = cfg['output_prefix']

# Resolve lora checkpoint to absolute path if given
if LORA_CHECKPOINT:
    lora_path = Path(LORA_CHECKPOINT)
    if not lora_path.is_absolute():
        lora_path = PROJECT_ROOT / lora_path
    assert lora_path.exists(), f'LoRA checkpoint not found: {lora_path}'
else:
    lora_path = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('model_id        :', MODEL_ID)
print('lora_checkpoint :', lora_path)
print('splits          :', SPLITS)
print('output_dir      :', OUTPUT_DIR)
print('output_prefix   :', OUTPUT_PREFIX)

## 2. Load Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

print(f'Loading tokenizer: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading base model: {MODEL_ID}')
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map='auto',
    trust_remote_code=True,
)

if lora_path:
    print(f'Loading LoRA adapter: {lora_path}')
    model = PeftModel.from_pretrained(model_base, str(lora_path))
else:
    print('No LoRA adapter — evaluating base model')
    model = model_base

model.eval()
DEVICE = next(model.parameters()).device
print('Model loaded on:', DEVICE)

## 3. Run Evaluation

In [ ]:
from datasets import load_dataset
from evaluate import run_evaluation, save_results, print_summary, print_wrong_examples

all_summaries = {}

for split in SPLITS:
    print(f'\n── Evaluating split: {split} ──────────────────────────')
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=split)
    print(f'Dataset size: {len(dataset)} examples')

    summary, results = run_evaluation(
        model=model,
        tokenizer=tokenizer,
        dataset=dataset,
        device=DEVICE,
        batch_size=EVAL_BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        desc=f'{OUTPUT_PREFIX} [{split}]',
    )

    # Add metadata to summary
    summary['model']           = MODEL_ID
    summary['lora_checkpoint'] = str(lora_path) if lora_path else 'base_model'
    summary['split']           = split

    output_path = OUTPUT_DIR / f'{OUTPUT_PREFIX}_{split}.json'
    save_results(summary, results, output_path)
    print_summary(f'{OUTPUT_PREFIX} [{split}]', summary)
    print_wrong_examples(results, n=3)

    all_summaries[split] = summary

## 4. Save Results to Google Drive

In [ ]:
import shutil

DRIVE_RESULTS = Path('/content/drive/MyDrive/gsm8k_eval_results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

for split in SPLITS:
    src = OUTPUT_DIR / f'{OUTPUT_PREFIX}_{split}.json'
    dst = DRIVE_RESULTS / src.name
    shutil.copy2(src, dst)
    print('Saved to Drive:', dst)